# YOLOv11 Segmentation Guide

**Author:** [Your Name/ID]  
**Date:** [Current Date]

## 1. Introduction

This notebook serves as a comprehensive guide to training a **YOLOv11** (You Only Look Once) model for **instance segmentation** tasks. YOLOv11 is the latest iteration in the YOLO family, developed by Ultralytics, offering state-of-the-art performance in real-time object detection and segmentation.

### Objectives
1.  **Environment Setup**: Install necessary libraries and verify GPU availability.
2.  **Data Preparation**: Understand the dataset structure required for YOLO.
3.  **Model Training**: Train a YOLOv11-Large segmentation model on a custom dataset.
4.  **Evaluation**: Analyze training metrics and validate model performance.
5.  **Inference**: Use the trained model to predict on new images.

---


## 2. Environment Setup

We use the `ultralytics` package, which provides the YOLOv11 implementation. We also check for CUDA support to ensure we are utilizing the GPU for faster training.

### 2.1 Install Dependencies & Import Libraries

In [ ]:
# Install the ultralytics package to access YOLO models
!pip install ultralytics

import torch
from ultralytics import YOLO
from IPython.display import Image, display
import os

# Check if CUDA (GPU support) is available.
# Developing on GPU is significantly faster than CPU.
print(f"Setup complete. Using torch {torch.__version__} ({'CUDA available' if torch.cuda.is_available() else 'CPU mode'})")

---
## 3. Data Preparation

To train YOLO, your dataset must be organized in a specific format. The directory structure typically looks like this:

```
dataset/
├── data.yaml       # Configuration file pointing to images and defining classes
├── train/
│   ├── images/     # Training images
│   └── labels/     # Corresponding labels (txt files)
├── val/
│   ├── images/     # Validation images
│   └── labels/     # Corresponding labels
└── test/           # (Optional) Test images
```

### The `data.yaml` File
This file is critical. It defines the paths to your training and validation sets, the number of classes, and their names.

**Example content of data.yaml:**
```yaml
train: ../train/images
val: ../val/images

nc: 1           # Number of classes
names: ['palm'] # Class names
```

### 3.1 Configure Dataset Path

In [ ]:
# Define the path to your data.yaml file
# This file contains paths to train/val images and class names.
# UPDATE THIS PATH to match your specific environment location (e.g. Google Drive path)
dataset_yaml_path = '/content/drive/MyDrive/Dec31_Dataset/version_1/data.yaml'

# Verify that the file exists before proceeding to avoid errors later
if not os.path.exists(dataset_yaml_path):
    print(f"WARNING: Dataset file not found at {dataset_yaml_path}. Please update the path.")
else:
    print(f"Dataset configuration found at: {dataset_yaml_path}")

---
## 4. Model Initialization

We will load a pre-trained Model. YOLOv11 comes in various sizes:
*   `yolo11n` (Nano): Fastest, least accurate.
*   `yolo11s` (Small)
*   `yolo11m` (Medium)
*   `yolo11l` (Large): High accuracy, slower.
*   `yolo11x` (Extra Large): Highest accuracy, slowest.

For this guide, we are using the **Large Segmentation** model (`yolo11l-seg.pt`).

### 4.1 Load YOLOv11 Model

In [ ]:
# Load the pre-trained YOLOv11 Large Segmentation model.
# 'yolo11l-seg.pt' indicates:
# - 'yolo11': Model version
# - 'l': Large size (more accurate but slower than 'n' or 's')
# - 'seg': Segmentation task (outputting masks, not just boxes)
# If the weights file is not found locally, it will be automatically downloaded.
model = YOLO("yolo11l-seg.pt")

# Display model information (layers, parameters, etc.)
model.info()

---
## 5. Training

We use the `model.train()` method to start training. Below is a breakdown of the key arguments used:

*   `data`: Path to the data configuration file.
*   `epochs`: Number of full passes through the dataset.
*   `imgsz`: Input image size (pixels). YOLO resizes images to this dimension.
*   `batch`: Batch size. reduce this if you run out of GPU memory.
*   `project`: Directory name to save results.
*   `name`: Sub-directory for this specific experiment.
*   `device`: The GPU device ID (e.g., 0).
*   `save`: Whether to save model checkpoints.

### 5.1 Execute Training Loop

In [ ]:
# Start the training process using the .train() method.
results = model.train(
    data=dataset_yaml_path,     # Path to data.yaml config
    epochs=200,                 # Total training epochs (iterations over the full dataset)
    imgsz=640,                  # Input image size (resized to 640x640)
    batch=4,                    # Batch size: Number of images processed per step. Lower this if you get CUDA OOM errors.
    device=0,                   # Device ID: 0 for the first GPU. Use 'cpu' for CPU training (very slow).
    project='oil_palm_thesis',  # Name of the project folder where results are saved
    name='yolo11l_Seg',         # Name of the specific experiment sub-folder
    save=True,                  # Save the best and last model checkpoints (weights)
    val=True,                   # Perform validation after each epoch to track performance
    plots=True                  # Generate training plots (loss curves, precision-recall, etc.)
)

---
## 6. Evaluation & Validation

After training, it is crucial to evaluate the model's performance on the validation set. Key metrics include:

*   **Box Loss**: Accuracy of bounding box coordinates.
*   **Seg Loss**: Accuracy of the segmentation masks.
*   **mAP50**: Mean Average Precision at IoU threshold 0.5.
*   **mAP50-95**: Average of mAP across IoU thresholds from 0.5 to 0.95 (a stricter metric).

The training process automatically generates plots in the project directory.

### 6.1 Calculate Validation Metrics

In [ ]:
# Run validation on the validation set explicitly using .val()
# This calculates metrics like mAP (mean Average Precision) on data the model hasn't trained on.
metrics = model.val()

# Print specific metrics for quick checking
# map50-95: The primary metric for COCO, average mAP over IoU thresholds 0.5 to 0.95
print(f"Maps (mAP50-95): {metrics.box.map}")
# map50: mAP at a lenient IoU threshold of 0.5
print(f"Maps50 (mAP50): {metrics.box.map50}")

### 6.2 Visualize Training Results

In [ ]:
# Display the training results graph generated by YOLO.
# This image shows the loss curves and metric improvements over epochs.
# The path is constructed from the 'project' and 'name' arguments used in training.
results_path = 'oil_palm_thesis/yolo11l_Seg/results.png'

if os.path.exists(results_path):
    display(Image(results_path))
else:
    print("Results plot not found. Check the save directory.")

### 6.3 Visualize Confusion Matrix

In [ ]:
# Display the confusion matrix to analyze misclassifications.
# Useful to see if the model is confusing background with objects or different classes with each other.
cm_path = 'oil_palm_thesis/yolo11l_Seg/confusion_matrix.png'

if os.path.exists(cm_path):
    display(Image(cm_path))
else:
    print("Confusion matrix not found.")

---
## 7. Inference (Prediction)

Now we can use the best trained text weights to predict on new images.

### 7.1 Run Predictive Inference

In [ ]:
# Load the best model weights saved during training.
# 'best.pt' contains the weights that achieved the highest mAP on the validation set.
best_model_path = 'oil_palm_thesis/yolo11l_Seg/weights/best.pt'

if os.path.exists(best_model_path):
    # Re-initialize the model with the best trained weights
    model = YOLO(best_model_path)
else:
    print("Best model weights not found. Using current model state.")

# Perform inference on a test image.
# Replace "path/to/your/test_image.jpg" with a real image path.
# conf=0.5: Only show detections with specific confidence score > 50%
# save=True: Save the resulting image with drawn bounding boxes/masks to disk
# results = model.predict("path/to/your/test_image.jpg", save=True, conf=0.5)

# To visualize the results interactively in the notebook:
# for result in results:
#    result.show()  # Display to screen (pop-up) or inline depending on environment

---
## Appendix A: Hyperparameters Reference

Here is a glossary of common training hyperparameters used in YOLO:

| Parameter | Description |
| :--- | :--- |
| `lr0` | Initial learning rate (i.e. SGD=1E-2, Adam=1E-3). |
| `lrf` | Final learning rate (lr0 * lrf). |
| `momentum` | SGD momentum/Adam beta1. |
| `weight_decay` | Optimizer weight decay (e.g. 5e-4). |
| `warmup_epochs` | Comparison epochs (fractions ok). |
| `box` | Box loss gain. |
| `cls` | Class loss gain (scale with pixels). |
| `dfl` | Distribution Focal Loss gain. |

## Appendix B: Troubleshooting

1.  **CUDA Out of Memory**: Reduce the `batch` size (e.g., from 16 to 8 or 4).
2.  **Model not learning (Loss doesn't decrease)**: Check your `datasets.yaml` and labels. Ensure labels are normalized (0-1) and correct.
3.  **Path Errors**: Ensure absolute paths are used in `data.yaml` if relative paths are confusing the loader.